# Translate Akkadian to English
This project aims to find out which model is best for translating an ancient language to contemporary English.

As the text samples are fragmented, have different formats and are relatively reduced in number (1500 entries), I looked at the ByT5 architecture, but one question still remains, and that is: which size should I choose?

Note that this code was designed to run on the kaggle interface notebooks, which can be pretty difficult to work on (12 hours max. run time and hardware limitations).

## Libraries and Data Preprocessing


In [2]:
!pip install datasets transformers evaluate accelerate sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.1 MB/s eta 0:00:00


In [20]:
import pandas as pd
import numpy as np
import torch
import os
import re
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback
)
import evaluate
import math
import kagglehub
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import gc

In [4]:
print("Downloading competition data")
input_path = kagglehub.competition_download('deep-past-initiative-machine-translation')
print("File path:", input_path)

train = pd.read_csv(os.path.join(input_path, 'train.csv'))
test = pd.read_csv(os.path.join(input_path, 'test.csv'))

print(train.head())

def clean_akkadian(text):
    if not isinstance(text, str): return ""

    repls = {'á': 'a2', 'à': 'a3', 'é': 'e2', 'è': 'e3', 'í': 'i2', 'ì': 'i3', 'ú': 'u2', 'ù': 'u3',
             'š': 'sz', 'Š': 'SZ', 'ṣ': 's,', 'Ṣ': 'S,', 'ṭ': 't,', 'Ṭ': 'T,', 'ḫ': 'h', 'Ḫ': 'H',
             'KÙ.B.': 'KÙ.BABBAR'}
    for char, rep in repls.items():
        text = text.replace(char, rep)

    subscripts = str.maketrans("₀₁₂₃₄₅₆₇₈₉", "0123456789")
    text = text.translate(subscripts)

    frac_map = {"0.5": "½", "0.25": "¼", "0.3333": "⅓", "0.8333": "⅚",
                "0.625": "⅝", "0.6666": "⅔", "0.75": "¾", "0.1666": "⅙"}
    for dec, frac in frac_map.items():
        text = text.replace(dec, frac)

    text = re.sub(r'\[x\]', '<gap>', text)
    text = re.sub(r'\[\.\.\.\]|\.\.\.', '<gap>', text)
    text = re.sub(r'[!?/\\:\[\]˹˺]', '', text)
    text = re.sub(r'<<|>>|<|>', '', text)

    return re.sub(r'\s+', ' ', text).strip()

def clean_english(text):
    if not isinstance(text, str): return ""

    frac_map = {"0.5": "½", "0.25": "¼", "0.3333": "⅓", "0.8333": "⅚",
                "0.625": "⅝", "0.6666": "⅔", "0.75": "¾", "0.1666": "⅙"}
    for dec, frac in frac_map.items():
        text = text.replace(dec, frac)

    roman_months = {
        "XII": "12", "XI": "11", "IX": "9", "VIII": "8", "VII": "7",
        "VI": "6", "IV": "4", "V": "5", "III": "3", "II": "2", "X": "10", "I": "1"
    }
    def month_replacer(match):
        return f"month {roman_months[match.group(1)]}"
    text = re.sub(r'\bmonth\s+(XII|XI|IX|VIII|VII|VI|IV|V|III|II|X|I)\b', month_replacer, text)

    text = text.replace("PN", "<gap>")
    text = text.replace("-gold", "pašallum gold")
    text = text.replace("-tax", "šaddūātum tax")
    text = text.replace("-textiles", "kutānum textiles")
    text = text.replace("1 / 12 (shekel)", "15 grains")
    text = text.replace("5 / 12 shekel", "⅓ shekel 15 grains")
    text = text.replace("5 11 / 12 shekels", "6 shekels less 15 grains")
    text = text.replace("7 / 12 shekel", "½ shekel 15 grains")

    terms_to_remove = ['fem.', 'sing.', 'pl.', 'plural', '(?)']
    for term in terms_to_remove:
        text = text.replace(term, "")

    text = re.sub(r'\s+/\s+[a-zA-Z]+', '', text)
    text = text.replace("<gap>", "MAGIC_GAP_TOKEN")
    text = re.sub(r'\.{2,}', '', text)
    text = re.sub(r'\bxx?\b', '', text)
    text = re.sub(r'<<|>>|<|>', '', text)
    text = text.replace("MAGIC_GAP_TOKEN", "<gap>")

    return re.sub(r'\s+', ' ', text).strip()

train['src'] = train['transliteration'].apply(clean_akkadian)
train['tgt'] = train['translation'].apply(clean_english)
test['src'] = test['transliteration'].apply(clean_akkadian)

train_data, val_data = train_test_split(train, test_size=0.1, random_state=42)

raw_datasets = DatasetDict({
    "train": Dataset.from_pandas(train_data),
    "validation": Dataset.from_pandas(val_data),
    "test": Dataset.from_pandas(test)
})


100%|██████████| 218M/218M [00:06<00:00, 34.8MB/s]

Extracting files...


File path: /root/.cache/kagglehub/competitions/deep-past-initiative-machine-translation
                                oare_id  \
0  004a7dbd-57ce-46f8-9691-409be61c676e   
1  0064939c-59b9-4448-a63d-34612af0a1b5   
2  0073f2c0-524c-4bbf-915a-8c1772a4fb98   
3  009fb838-8038-42bc-ad34-5f795b3840ee   
4  00aa1c55-c80c-4346-a159-73ad43ab0ff7   

                                     transliteration  \
0  KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-{d}IM KIŠI...   
1               1 TÚG ša qá-tim i-tur₄-DINGIR il₅-qé   
2  TÚG u-la i-dí-na-ku-um i-tù-ra-ma 9 GÍN KÙ.BABBAR   
3  KIŠIB šu-{d}EN.LÍL DUMU šu-ku-bi-im KIŠIB ṣí-l...   
4  um-ma šu-ku-tum-ma a-na IŠTAR-lá-ma-sí ù ni-ta...   

                                         translation  
0  Seal of Mannum-balum-Aššur son of Ṣilli-Adad, ...  
1  Itūr-ilī has received one textile of ordinary ...  
2  <gap> he did not give you a textile. He return...  
3  Seal of Šu-Illil son of Šu-Kūbum, seal of Ṣilū...  
4  From Šukkutum to Ištar-lamassī and Nit

Now that the data has been processed according to the official competition data page, we can move on to model selection and tokenization.

In [18]:
model_checkpoint = "google/byt5-large" #for model selection just change this line between small base and large
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 512

def preprocess_function(examples):
    prefix = "translate Akkadian to English: "
    inputs = [prefix + doc for doc in examples["src"]]
    model_inputs = tokenizer(inputs, max_length=MAX_INPUT_LENGTH, truncation=True)

    if "tgt" in examples:
        targets = examples["tgt"]
        labels = tokenizer(text_target=targets, max_length=MAX_TARGET_LENGTH, truncation=True)
        model_inputs["labels"] = labels["input_ids"]

    return model_inputs

tokenized_train = raw_datasets["train"].map(
    preprocess_function,
    batched=True,
    remove_columns=raw_datasets["train"].column_names
)

tokenized_val = raw_datasets["validation"].map(
    preprocess_function,
    batched=True,
    remove_columns=raw_datasets["validation"].column_names
)

tokenized_test = raw_datasets["test"].map(
    preprocess_function,
    batched=True,
    remove_columns=raw_datasets["test"].column_names
)

tokenized_datasets = DatasetDict({
    "train": tokenized_train,
    "validation": tokenized_val,
    "test": tokenized_test
})

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.59k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.50k [00:00<?, ?B/s]

Map:   0%|          | 0/1404 [00:00<?, ? examples/s]

Map:   0%|          | 0/157 [00:00<?, ? examples/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

## Evaluation Metric

During training, I used the official competition metric to ensure consistency with the final scoring on the hidden test set.

In [6]:
metric_bleu = evaluate.load("sacrebleu")
metric_chrf = evaluate.load("chrf")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]

    bleu_result = metric_bleu.compute(predictions=decoded_preds, references=decoded_labels)
    bleu_score = bleu_result["score"]

    chrf_result = metric_chrf.compute(predictions=decoded_preds, references=decoded_labels, word_order=2)
    chrf_score = chrf_result["score"]

    if bleu_score > 0 and chrf_score > 0:
        geom_mean = math.sqrt(bleu_score * chrf_score)
    else:
        geom_mean = 0.0

    return {
        "bleu": bleu_score,
        "chrf": chrf_score,
        "geom_mean": geom_mean
    }

## SMALL BYT5 MODEL

A model with aprox. 300M parameters means that we can use basically every trick that we know with this one and the memory won't be a problem.

In [12]:
model_name = model_checkpoint.split("/")[-1]
output_dir = f"{model_name}-finetuned-akkadian-to-english"
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-3,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    label_smoothing_factor=0.1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=15,
    predict_with_generate=False,
    bf16=True,
    push_to_hub=False,

    load_best_model_at_end=True,
    metric_for_best_model="eval_geom_mean",
    greater_is_better=True
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

Loading weights:   0%|          | 0/172 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Everything is set up for training, but be prepared to wait for a while (around 4.5 hours for the small model). After training, the model should be in the byt5-small file.

In [13]:
#trainer.train()

Now that we have a model, let's start generating some predictions.

In [ ]:
MODEL_PATH = output_dir

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

prefix = "translate Akkadian to English: "
test_texts = [prefix + text for text in test['src'].tolist()]
test_ids = test['id'].tolist()

MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 512
batch_size = 16
predictions = []

for i in tqdm(range(0, len(test_texts), batch_size)):
    batch_texts = test_texts[i : i + batch_size]

    inputs = tokenizer(
        batch_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_LENGTH
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_TARGET_LENGTH,
            num_beams=12,
            length_penalty=0.9,
            no_repeat_ngram_size=18,
            repetition_penalty=1.4,
            early_stopping=True
        )

    batch_preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    predictions.extend(batch_preds)

submission_df = pd.DataFrame({
    'id': test_ids,
    'translation': predictions
})

submission_df.to_csv('submission_small.csv', index=False)
print("Submission file fully generated with the small model")

This model got a score of 23.94.

## BASE BYT5 MODEL

This time the model has around 580M parameters so we need to make some small adjustments to the fine-tuning hyperparameters.

In [15]:
model_name = model_checkpoint.split("/")[-1]
output_dir = f"{model_name}-finetuned-akkadian-to-english"

model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    save_strategy="epoch",

    # Lower learning rate (2e-4 instead of 1e-3).
    # Larger models are more sensitive => they require smaller steps to maintain mathematical stability and avoid diverging during training.
    learning_rate=2e-4,

    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    label_smoothing_factor=0.1,

    # Batch size drastically reduced to 1 (from 2).
    # The Base model takes up more VRAM => educing the physical batch size to 1 is mandatory to prevent CUDA Out-Of-Memory errors.
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    # Gradient accumulation doubled to 16 (from 8).
    # This compensates for the batch_size=1, keeping the mathematical batch size exactly the same as the Small model (1 * 16 = 16).
    gradient_accumulation_steps=16,

    weight_decay=0.01,

    # Limit reduced to 1 (from 3) and save_only_model=True added.
    # Base model checkpoints are massive => prevents disks from filling up by keeping only the single best checkpoint and discarding optimizer states.
    save_total_limit=1,
    save_only_model=True,

    num_train_epochs=15,
    predict_with_generate=True,

    # Max length cut in half to 256 (from 512).
    # Capping the generation length speeds up the evaluation phase and controls memory spikes.
    generation_max_length=256,

    # Gradient checkpointing enabled (True).
    # It trades compute for memory by dropping intermediate activations and recomputing them during the backward pass.
    gradient_checkpointing=True,

    # Mixed precision completely disabled (fp16=False, bf16=False).
    # While slower and more memory-intensive, it completely eliminates the risk of numerical overflows (NaN).
    fp16=False,
    bf16=False,

    push_to_hub=False,
    load_best_model_at_end=True,
    metric_for_best_model="eval_geom_mean",
    greater_is_better=True
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,

    # Patience increased to 3 (from 2).
    # Gives the Base model a slightly longer time to escape local minimums or plateaus before early stopping terminates the training.
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

pytorch_model.bin:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


model.safetensors:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [16]:
#trainer.train()

It's time to generate another set of predictions, but this time I added a post-processing function to try and make better English sentences.

In [ ]:
MODEL_PATH = output_dir

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

# Added `tie_word_embeddings=False`.
# This is explicitly required for Base model to load properly, preventing size mismatches between encoder and decoder embeddings.
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH, tie_word_embeddings=False)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

def post_process_translation(text):
    if not text: return ""
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'\s+([.,?!])', r'\1', text)
    text = re.sub(r"\s+'\s+", "'", text)
    text = re.sub(r"\s+'", "'", text)
    text = re.sub(r'\.{2,}', '.', text)
    text = re.sub(r',{2,}', ',', text)
    text = text.replace('< gap >', '<gap>').replace(' <gap>', '<gap>')
    if len(text) > 0:
        text = text[0].upper() + text[1:]
    if len(text) > 0 and text[-1] not in ['.', '!', '?', '"', "'"]:
        text += '.'
    return text

prefix = "translate Akkadian to English: "
test_texts = [prefix + text for text in test['src'].tolist()]
test_ids = test['id'].tolist()

MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 512

# Inference batch size drastically reduced from 16 to 4.
# The Base model requires much more VRAM during generation, especially when paired with a huge Beam Search parameter.
batch_size = 4
predictions = []

for i in tqdm(range(0, len(test_texts), batch_size)):
    batch_texts = test_texts[i : i + batch_size]

    inputs = tokenizer(
        batch_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_LENGTH
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_TARGET_LENGTH,

            # Beam Search (30 instead of 12).
            # Forces the model to explore 30 parallel translation paths to find the most grammatically accurate sentence, maximizing the BLEU score at the cost of speed.
            num_beams=30,

            # Positive length penalty (1.30 instead of 0.9).
            # Small model was penalized for long texts to stop hallucinations.
            # Base model is smarter, so we actively encourage it to generate longer, more complete translations.
            length_penalty=1.30,

            # Relaxed repetition parameters.
            # no_repeat_ngram_size increased to 25, and repetition_penalty dropped to 1.08.
            # Allows the Base model to naturally repeat administrative formulas if needed, without being artificially blocked by the algorithm.
            no_repeat_ngram_size=25,
            repetition_penalty=1.08,
            early_stopping=True
        )

    batch_preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    # Applying the post-processing filter on every generated batch.
    batch_preds = [post_process_translation(p) for p in batch_preds]
    predictions.extend(batch_preds)

submission_df = pd.DataFrame({
    'id': test_ids,
    'translation': predictions
})

submission_df.to_csv('submission_base.csv', index=False)
print("Submission file fully generated with the base model")

This was by far the best solution generated with a score of 29.71.

## LARGE BYT5 MODEL

Running a 1.2B parameters model with such time and hardware contrains was a challange. Another problem for this size was the actual training and validation data. As you can recall the dataset doesn't have a lot of samples, which means that a large model can easily overfit.

In [19]:
model_name = model_checkpoint.split("/")[-1]
output_dir = f"{model_name}-finetuned-akkadian-to-english"

model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    save_strategy="epoch",

    # Significantly lower learning rate (5e-5 vs 2e-4).
    # Massive models suffer from gradient instability and parameter shock. They require really small update steps to avoid diverging.
    learning_rate=5e-5,

    lr_scheduler_type="cosine",

    # Switched from `warmup_ratio` to explicit `warmup_steps=150`.
    # This provides strict, absolute control over the warmup phase, ensuring the massive model doesn't break in early batches.
    warmup_steps=150,

    label_smoothing_factor=0.1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    gradient_checkpointing=True,
    fp16=False,
    fp16_full_eval=False,

    # Changed optimizer to 'adafactor' (Default is AdamW).
    # Adafactor is explicitly designed to save VRAM for large T5/ByT5 models by dropping rolling momentum components.
    # Without this, the Large model would instantly trigger a CUDA Out-Of-Memory error.
    optim="adafactor",

    weight_decay=0.01,
    save_total_limit=1,
    save_only_model=True,

    # Epochs reduced to 10 (from 15).
    # The immense capacity of the Large model means it memorizes the small 1500-entry dataset much faster.
    num_train_epochs=10,

    # Added `dataloader_num_workers=2`.
    # Speeds up CPU data loading to keep the GPU constantly fed, avoiding bottlenecks during the heavy training steps.
    dataloader_num_workers=2,

    # generative validation turned OFF (False instead of True).
    # This was turned off to survive the strict 12-hour Kaggle kernel limit.
    predict_with_generate=False,

    push_to_hub=False,
    load_best_model_at_end=True,

    # Reverted to "eval_loss" instead of BLEU/geom_mean.
    # Because `predict_with_generate` is False, we can only evaluate mathematical loss.
    # This is the exact reason the Large model suffered from overfitting, it optimized for memorizing training keys
    # rather than actual translation quality.
    metric_for_best_model="eval_loss",
    greater_is_better=False
)

# Removed `compute_metrics` entirely.
# Since generation is off, we cannot compute BLEU/chrF during training anymore.
trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,

    # Early stopping patience reduced to 2 (from 3).
    # The Large model overfits aggressively, so we must stop it as fast as we can.
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

pytorch_model.bin:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/500 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
#trainer.train()

Now it's time to generate the final solution (spoiler: it won't be a good one):

In [ ]:
MODEL_PATH = output_dir

# local_files_only=True ensures the tokenizer doesn't
# attempt to connect to the HuggingFace hub, forcing it to use your massive
# locally saved checkpoint from the Kaggle/Colab output directory.
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)

# Added `torch_dtype=torch.float16`.
# The Large model cannot fit into a standard 16GB GPU in FP32. Forcing it into Half-Precision (FP16) during inference
# is required to avoid an immediate CUDA Out-Of-Memory error.
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_PATH,
    tie_word_embeddings=False,
    local_files_only=True,
    torch_dtype=torch.float16
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

def post_process_translation(text):
    if not text: return ""
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'\s+([.,?!])', r'\1', text)
    text = re.sub(r"\s+'\s+", "'", text)
    text = re.sub(r"\s+'", "'", text)
    text = re.sub(r'\.{2,}', '.', text)
    text = re.sub(r',{2,}', ',', text)
    text = text.replace('< gap >', '<gap>').replace(' <gap>', '<gap>')
    if len(text) > 0:
        text = text[0].upper() + text[1:]
    if len(text) > 0 and text[-1] not in ['.', '!', '?', '"', "'"]:
        text += '.'
    return text

prefix = "translate Akkadian to English: "
test_texts = [prefix + text for text in test['src'].tolist()]
test_ids = test['id'].tolist()

MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 512

# Batch size reduced by half (from 4 down to 2).
# Even in FP16, the generative context window of the Large model eats up VRAM rapidly.
batch_size = 2
predictions = []

for i in tqdm(range(0, len(test_texts), batch_size)):
    batch_texts = test_texts[i : i + batch_size]

    inputs = tokenizer(
        batch_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_LENGTH
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_TARGET_LENGTH,

            # Beam search reduced to 20 (from 30).
            # The Base model could afford 30 parallel beams. The Large model would crash the memory or timeout if pushed that far.
            num_beams=20,

            length_penalty=1.28,
            no_repeat_ngram_size=25,

            # Repetition penalty increased to 1.15 (from 1.08).
            # Because the Large model was overfitted, it has a severe tendency to get stuck in
            # hallucination loops on unseen text. This higher penalty actively forces the decoder to stop repeating the same memorized phrases.
            repetition_penalty=1.15,

            early_stopping=True
        )

    batch_preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    batch_preds = [post_process_translation(p) for p in batch_preds]
    predictions.extend(batch_preds)

submission_df = pd.DataFrame({
    'id': test_ids,
    'translation': predictions
})

submission_df.to_csv('submission_large.csv', index=False)
print("Submission file fully generated with the large model")

As I said earlier, this was a bad submission, with a score of 20.57. Even after everything I tried to stop this, the model still overfitted on the training data, due to the lack of samples.

## Solution Analysis

With the following code I tried to see for each model which sentences were good, which were bad and wheter or not there was a pattern.

In [ ]:
#just choose the model that you want to run the cell for between small base and large
MODEL_TYPE = "large"

MODEL_PATH = f"/content/byt5-{MODEL_TYPE}-finetuned-akkadian-to-english"

if MODEL_TYPE == "small":
    BATCH_SIZE = 16
    NUM_BEAMS = 12
    USE_FP16 = False
elif MODEL_TYPE == "base":
    BATCH_SIZE = 4
    NUM_BEAMS = 30
    USE_FP16 = False
elif MODEL_TYPE == "large":
    BATCH_SIZE = 2
    NUM_BEAMS = 20
    USE_FP16 = True

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)

if USE_FP16:
    model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_PATH,
        tie_word_embeddings=False,
        local_files_only=True,
        torch_dtype=torch.float16
    )
else:
    model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_PATH,
        tie_word_embeddings=False,
        local_files_only=True
    )

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

def post_process_translation(text):
    if not text: return ""
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'\s+([.,?!])', r'\1', text)
    text = re.sub(r"\s+'\s+", "'", text)
    text = re.sub(r"\s+'", "'", text)
    text = re.sub(r'\.{2,}', '.', text)
    text = re.sub(r',{2,}', ',', text)
    text = text.replace('< gap >', '<gap>').replace(' <gap>', '<gap>')
    if len(text) > 0:
        text = text[0].upper() + text[1:]
    if len(text) > 0 and text[-1] not in ['.', '!', '?', '"', "'"]:
        text += '.'
    return text

prefix = "translate Akkadian to English: "
test_texts = [prefix + text for text in val_data['src'].tolist()]
references_english = val_data['tgt'].tolist()
val_ids = val_data['id'].tolist() if 'id' in val_data.columns else range(len(test_texts))

predictions = []
for i in tqdm(range(0, len(test_texts), BATCH_SIZE)):
    batch_texts = test_texts[i : i + BATCH_SIZE]

    inputs = tokenizer(
        batch_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            num_beams=NUM_BEAMS,
            length_penalty=1.30 if MODEL_TYPE != "small" else 0.9,
            no_repeat_ngram_size=25 if MODEL_TYPE != "small" else 18,
            repetition_penalty=1.08 if MODEL_TYPE != "small" else 1.4,
            early_stopping=True
        )

    batch_preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    batch_preds = [post_process_translation(p) for p in batch_preds]
    predictions.extend(batch_preds)

analysis_df = pd.DataFrame({
    'id': val_ids,
    'Akkadian_Original': val_data['transliteration'].tolist(),
    'Reference_Truth': references_english,
    f'{MODEL_TYPE.capitalize()}_Prediction': predictions
})

smoother = SmoothingFunction().method1

def calculate_sentence_bleu(row):
    ref = str(row['Reference_Truth']).lower().split()
    cand = str(row[f'{MODEL_TYPE.capitalize()}_Prediction']).lower().split()
    if len(cand) == 0:
        return 0.0
    return sentence_bleu([ref], cand, smoothing_function=smoother) * 100

analysis_df['Sentence_BLEU'] = analysis_df.apply(calculate_sentence_bleu, axis=1)
analysis_df['Length_Diff'] = analysis_df['Reference_Truth'].astype(str).str.len() - analysis_df[f'{MODEL_TYPE.capitalize()}_Prediction'].astype(str).str.len()

succese = analysis_df.sort_values(by='Sentence_BLEU', ascending=False).head(10)
esecuri = analysis_df.sort_values(by='Sentence_BLEU', ascending=True).head(10)

print("\n 10 BEST SENTENCES \n")
for _, row in succese.iterrows():
    print(f"\n[BLEU score: {row['Sentence_BLEU']:.2f}]")
    print(f"AKKADIAN: {row['Akkadian_Original']}")
    print(f"CORRECT:   {row['Reference_Truth']}")
    print(f"PREDICTED: {row[f'{MODEL_TYPE.capitalize()}_Prediction']}")

print("\n 10 WORST SENTENCES \n")
for _, row in esecuri.iterrows():
    print(f"\n[BLEU score: {row['Sentence_BLEU']:.2f} | Length dif: {row['Length_Diff']} chars]")
    print(f"AKKADIAN: {row['Akkadian_Original']}")
    print(f"CORRECT:   {row['Reference_Truth']}")
    print(f"PREDICTED: {row[f'{MODEL_TYPE.capitalize()}_Prediction']}")

csv_name = f'analysis_{MODEL_TYPE}_model.csv'
analysis_df.to_csv(csv_name, index=False)
print(f"Analysis file fully generated")

del model
torch.cuda.empty_cache()
gc.collect()

## Conclusion

We've seen that a bigger model isn't always better, as there are many variabels to take into consideration, most importantly, those regarding the training dataset (size, format, etc..).

While the small model could translate small administrative sentences, it struggled with large phrases due to the lack of memory. The base mode was the sweet spot between memory and dataset size, being big enough for complex sentences, but not big enough to overfit on training data. The large model was a complete disaster (taking a really long time to train and generating a bad submission) due to hardware and dataset constrains.